# FaceSwap Demo — Google Colab

**GitHub × Drive 構成:**
- ソースコード → GitHub から毎回最新を取得
- モデル・フレームキャッシュ → Drive に保存して使いまわし

**初回起動の手順:**
1. Cell 3 の `NGROK_TOKEN` を設定
2. 上から順に実行（初回はモデルDL で数分かかります）

**2回目以降は全セルをそのまま実行するだけ。**

In [ ]:
# Cell 1: GPU ランタイム確認
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('❌ GPU が検出されません！')
    print('→ Runtime > Change runtime type > T4 GPU を選択して再実行してください')
    raise RuntimeError('GPUランタイムが必要です')

In [ ]:
# Cell 2: Drive マウント & GitHub からコード取得
import os, subprocess, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_URL  = 'https://github.com/oyatuchokobi/faceswap.git'
REPO_DIR  = '/content/faceswap'
DRIVE_DIR = '/content/drive/MyDrive/faceswap-data'  # Drive 上の保存先

# --- GitHub クローン / プル ---
if Path(f'{REPO_DIR}/.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=True)
    print('✅ GitHub から最新コードを取得しました')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('✅ GitHub からクローンしました')

# --- Drive の保存ディレクトリを準備 ---
for d in ['models', 'templates/basketball']:
    Path(f'{DRIVE_DIR}/{d}').mkdir(parents=True, exist_ok=True)

# --- シンボリックリンク: models/ → Drive ---
models_link = Path(f'{REPO_DIR}/models')
if models_link.exists() and not models_link.is_symlink():
    shutil.rmtree(str(models_link))
if not models_link.is_symlink():
    models_link.symlink_to(f'{DRIVE_DIR}/models')

# --- シンボリックリンク: templates/basketball/ → Drive (フレームキャッシュ) ---
frames_link = Path(f'{REPO_DIR}/templates/basketball')
frames_link.parent.mkdir(parents=True, exist_ok=True)
if frames_link.exists() and not frames_link.is_symlink():
    shutil.rmtree(str(frames_link))
if not frames_link.is_symlink():
    frames_link.symlink_to(f'{DRIVE_DIR}/templates/basketball')

print(f'✅ Drive 連携完了')
print(f'   models/      → {DRIVE_DIR}/models')
print(f'   template frames → {DRIVE_DIR}/templates/basketball')

In [ ]:
# Cell 3: 依存パッケージ & ngrok 設定
import subprocess, sys

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'  # ← https://dashboard.ngrok.com/get-started/your-authtoken

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi==0.115.0',
    'uvicorn[standard]==0.32.0',
    'python-multipart==0.0.12',
    'insightface==0.7.3',
    'onnxruntime-gpu==1.20.0',
    'numpy>=1.24,<2.0',
    'opencv-python-headless==4.10.0.84',
    'Pillow==11.0.0',
    'aiofiles==24.1.0',
    'sse-starlette==2.1.3',
    'pyngrok',
], check=True)
print('✅ インストール完了')

import onnxruntime as ort
providers = ort.get_available_providers()
if 'CUDAExecutionProvider' in providers:
    print(f'✅ GPU推論が有効 (CUDA)')
else:
    print(f'⚠️  CUDAなし → CPU推論になります。プロバイダー: {providers}')

In [ ]:
# Cell 4: モデル確認 / ダウンロード（初回のみ Drive に保存）
from pathlib import Path
import subprocess, sys

model = Path(f'{REPO_DIR}/models/inswapper_128.onnx')

if not model.exists():
    print('モデルをダウンロード中... (初回のみ、約 554MB、数分かかります)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    subprocess.run([
        'huggingface-cli', 'download',
        'ezioruan/inswapper_128.onnx', 'inswapper_128.onnx',
        '--local-dir', str(model.parent)
    ], check=True)
    print('✅ ダウンロード完了・Drive に保存しました（次回以降は不要）')
else:
    print(f'✅ モデル確認済み（Drive より使用: {model}）')

In [ ]:
# Cell 5: uvicorn をバックグラウンドで起動
import os, subprocess, sys, time, urllib.request

os.chdir(REPO_DIR)

proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.main:app',
     '--host', '0.0.0.0', '--port', '8000'],
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print('サーバー起動中... (初回はモデルロードで1〜2分かかります)')
for _ in range(24):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=3)
        print('✅ サーバー起動完了')
        break
    except Exception:
        print('  ...待機中')
else:
    print('⚠️  起動タイムアウト。ログ:')
    print(proc.stdout.read(3000).decode(errors='replace'))

In [ ]:
# Cell 6: ngrok トンネル → アプリ URL を表示
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000)

print(f'\n🎉 アプリURL: {tunnel.public_url}')
print('このURLをブラウザで開いてください（スマホも可）')
print(f'デバッグ用: {tunnel.public_url}/?mode=stg')